# NB15 — Calibration-gap figure for publication

Generates the final 2×2 calibration-gap diagram for the two leading FULL_11 models from NB14 outputs.

**Recommended runtime:** CPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT=Path('/content/drive/MyDrive/DRY_BEAN_HYBRID_Q1')
NB14=ROOT/'03_RESULTS'/'NB14_FINAL_PCA_LDA_CALIBRATION'
OUT=ROOT/'03_RESULTS'/'NB15_CALIBRATION_GAP_FIGURE'
FIG=ROOT/'05_FIGURES'/'FINAL_PUBLICATION'
OUT.mkdir(parents=True,exist_ok=True); FIG.mkdir(parents=True,exist_ok=True)
bins=pd.read_csv(NB14/'figure7_reliability_bins.csv')
val=pd.read_csv(NB14/'figure7_calibration_validation_by_seed.csv')
ece=val.groupby(['model','model_label'],as_index=False).agg(raw_ece=('raw_ece','mean'),temp_ece=('temp_ece','mean'))

In [ ]:
specs=[
 ('TabPFN','TabPFN','Raw','raw_ece','(a)','Raw'),
 ('TabPFN','TabPFN','Cross-fitted temperature','temp_ece','(b)','Temperature-scaled'),
 ('LDA_XGBoost','LDA-XGBoost','Raw','raw_ece','(c)','Raw'),
 ('LDA_XGBoost','LDA-XGBoost','Cross-fitted temperature','temp_ece','(d)','Temperature-scaled')]
ACC='#7EA6C4'; GAP='#FF7F7F'; EDGE='#666666'; MIN=10
plt.rcParams.update({'font.size':9,'axes.titlesize':10,'axes.labelsize':9,'legend.fontsize':8,'font.family':'serif'})
fig,axes=plt.subplots(2,2,figsize=(12.4,8.7),sharex=True,sharey=True); axes=axes.ravel()
manifest=[]
for ax,(mid,label,calname,ecol,panel,suffix) in zip(axes,specs):
    d=bins[(bins.model_id==mid)&(bins.calibration==calname)&(bins['count']>=MIN)].copy().sort_values('bin')
    ev=float(ece.loc[ece.model==mid,ecol].iloc[0])
    centers=(d.lower.to_numpy()+d.upper.to_numpy())/2; widths=(d.upper.to_numpy()-d.lower.to_numpy())*.90
    acc=d.accuracy.to_numpy(float); conf=d.mean_confidence.to_numpy(float)
    ax.bar(centers,acc,width=widths,color=ACC,edgecolor=EDGE,linewidth=.55,alpha=.95,label='Accuracy',zorder=2)
    lower=np.minimum(acc,conf); gap=np.abs(acc-conf)
    ax.bar(centers,gap,bottom=lower,width=widths,color=GAP,edgecolor=EDGE,linewidth=.45,alpha=.92,label='Gap',zorder=3)
    ax.plot([0,1],[0,1],'--',linewidth=1.05,color='black',label='Perfect calibration',zorder=4)
    ax.set_title(f'{panel} {label} — {suffix} (ECE={ev:.4f})',fontweight='bold')
    ax.set_xlabel('Confidence'); ax.set_ylabel('Accuracy'); ax.set_xlim(0,1); ax.set_ylim(0,1); ax.set_xticks(np.arange(0,1.01,.2)); ax.set_yticks(np.arange(0,1.01,.2)); ax.grid(True,alpha=.18,linewidth=.6,zorder=0)
    for _,r in d.iterrows(): manifest.append({'panel':panel,'model_id':mid,'model_label':label,'calibration':calname,'ece_mean_across_seeds':ev,'bin':int(r['bin']),'lower':float(r.lower),'upper':float(r.upper),'count':int(r['count']),'mean_confidence':float(r.mean_confidence),'accuracy':float(r.accuracy),'gap':float(abs(r.accuracy-r.mean_confidence))})
h,l=axes[0].get_legend_handles_labels(); order=[l.index('Perfect calibration'),l.index('Accuracy'),l.index('Gap')]
fig.legend([h[i] for i in order],[l[i] for i in order],loc='upper center',ncol=3,frameon=True,bbox_to_anchor=(.5,1.005))
fig.suptitle('Calibration of Leading FULL_11 Models',y=1.035,fontsize=12,fontweight='bold')
plt.tight_layout(rect=[0,0,1,.965])
stem=FIG/'Figure_7_Calibration_Gap_2x2_TabPFN_LDA_XGBoost'
plt.savefig(str(stem)+'.png',dpi=600,bbox_inches='tight'); plt.savefig(str(stem)+'.pdf',bbox_inches='tight'); plt.savefig(str(stem)+'.svg',bbox_inches='tight'); plt.show()
pd.DataFrame(manifest).to_csv(OUT/'figure7_gap_plot_manifest.csv',index=False); ece.to_csv(OUT/'figure7_gap_plot_ece_summary.csv',index=False)
print('NB15 complete.')